# MetaCal Benchmark — T-06

Isolated task notebook.

In [3]:
import re
import kaggle_benchmarks as kbench

def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """AUROC — how well confidence predicts correctness."""
    pairs = sorted(zip(confidences, correctness), reverse=True)
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    tp, fp, auc = 0, 0, 0
    prev_fp = 0
    for conf, correct in pairs:
        if correct:
            tp += 1
        else:
            fp += 1
            auc += tp * (fp - prev_fp)
            prev_fp = fp
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d_proxy(correct_confs, incorrect_confs):
    """Discrimination between correct and incorrect confidence."""
    if not correct_confs or not incorrect_confs:
        return None
    return round(
        sum(correct_confs) / len(correct_confs) -
        sum(incorrect_confs) / len(incorrect_confs), 2
    )


In [ ]:
@kbench.task(
    name="T-06: Contradiction Detection Under Paraphrase",
    description="Paired statements where one (or neither) contains a subtle semantic inversion. Model must identify which statement is wrong (A, B, or Neither) and report confidence."
)
def t06_contradiction_detection(llm) -> None:
    PAIRS = [
        # Statement B is wrong (7 pairs)
        (
            "Statement A: The Moon orbits Earth roughly every 27 days.",
            "Statement B: Earth completes one orbit around the Moon every 27 days.",
            "B", "Earth does not orbit the Moon — the Moon orbits Earth."
        ),
        (
            "Statement A: Antibiotics are effective against bacterial infections.",
            "Statement B: Antibiotics are effective against viral infections.",
            "B", "Antibiotics target bacteria, not viruses."
        ),
        (
            "Statement A: DNA stores genetic information in a double helix structure.",
            "Statement B: RNA stores genetic information in a double helix structure.",
            "B", "RNA is typically single-stranded; DNA is the double helix."
        ),
        (
            "Statement A: Light travels at its maximum speed in a vacuum.",
            "Statement B: Light travels at its maximum speed in water.",
            "B", "Light slows down in denser media such as water; it travels fastest in a vacuum."
        ),
        (
            "Statement A: Mammals are warm-blooded animals.",
            "Statement B: Mammals are cold-blooded animals.",
            "B", "Mammals regulate their own body temperature; they are warm-blooded."
        ),
        (
            "Statement A: Plants produce oxygen through photosynthesis.",
            "Statement B: Plants produce carbon dioxide through photosynthesis.",
            "B", "Photosynthesis converts CO2 into glucose and releases O2, not CO2."
        ),
        (
            "Statement A: The Pacific Ocean is the largest ocean on Earth.",
            "Statement B: The Atlantic Ocean is the largest ocean on Earth.",
            "B", "The Pacific is the largest ocean, covering more area than all landmasses combined."
        ),
        # Statement A is wrong (5 pairs)
        (
            "Statement A: The Sun orbits the Earth once every 365 days.",
            "Statement B: The Earth orbits the Sun once every 365 days.",
            "A", "It is the Earth that orbits the Sun, not the other way around."
        ),
        (
            "Statement A: Sound travels faster than light.",
            "Statement B: Light travels faster than sound.",
            "A", "Light travels at approximately 3x10^8 m/s; sound travels at approximately 343 m/s in air."
        ),
        (
            "Statement A: Humans have 48 chromosomes in each somatic cell.",
            "Statement B: Humans have 46 chromosomes in each somatic cell.",
            "A", "Humans have 46 chromosomes (23 pairs) in somatic cells, not 48."
        ),
        (
            "Statement A: Iron is denser than gold.",
            "Statement B: Gold is denser than iron.",
            "A", "Gold has a density of ~19.3 g/cm^3 vs iron's ~7.87 g/cm^3; gold is far denser."
        ),
        (
            "Statement A: The mitochondria is the control centre of the cell, housing the cell's DNA.",
            "Statement B: The nucleus is the control centre of the cell, housing the cell's DNA.",
            "A", "The nucleus is the control centre and contains most of the cell's DNA; the mitochondria produces energy."
        ),
        # Neither statement is wrong (3 pairs)
        (
            "Statement A: Water freezes at 0 deg C at standard atmospheric pressure.",
            "Statement B: Water boils at 100 deg C at standard atmospheric pressure.",
            "Neither", "Both statements are correct standard physical facts."
        ),
        (
            "Statement A: The Earth orbits the Sun.",
            "Statement B: The Moon orbits the Earth.",
            "Neither", "Both statements are correct astronomical facts."
        ),
        (
            "Statement A: Gold has the atomic number 79.",
            "Statement B: Silver has the atomic number 47.",
            "Neither", "Both are correct: Au=79 and Ag=47 are standard periodic table values."
        ),
    ]
    PROMPT_TEMPLATE = (
        "{a}\n{b}\n\n"
        "Which statement, if any, contains a factual error? Answer A, B, or Neither. "
        "Then state your confidence 0-100."
    )

    for stmt_a, stmt_b, wrong_stmt, explanation in PAIRS:
        response = llm.prompt(PROMPT_TEMPLATE.format(a=stmt_a, b=stmt_b))
        conf = extract_confidence(response)
        kbench.assertions.assert_in(
            wrong_stmt, response,
            expectation=f"Model must identify '{wrong_stmt}' as the answer. Reason: {explanation}"
        )
        kbench.assertions.assert_true(
            conf is not None,
            expectation="Model must output a numeric confidence score 0-100."
        )

In [ ]:
%choose t06_contradiction_detection